### Import libraries and set paths

In [5]:
from pathlib import Path
import joblib
import pandas as pd

from scipy.sparse import load_npz
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

FEATURES_DIR = ROOT / "data" / "interim" / "tfidf_features"
MODEL_DIR = ROOT / "models" / "traditional_ml"
OUTPUT_DIR = ROOT / "outputs" / "evaluation"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)

Project root: d:\My_Project\Analyse\mobile-review-sentiment-analysis


### Load TF-IDF features and labels

In [6]:
X_train = load_npz(FEATURES_DIR / "X_train_tfidf.npz")
X_val = load_npz(FEATURES_DIR / "X_validation_tfidf.npz")
X_test = load_npz(FEATURES_DIR / "X_test_tfidf.npz")

y_train = pd.read_csv(FEATURES_DIR / "y_train.csv")["sentiment"].astype(str).str.strip().str.lower()
y_val = pd.read_csv(FEATURES_DIR / "y_validation.csv")["sentiment"].astype(str).str.strip().str.lower()
y_test = pd.read_csv(FEATURES_DIR / "y_test.csv")["sentiment"].astype(str).str.strip().str.lower()

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)
assert X_test.shape[0] == len(y_test)

assert X_train.shape[1] == X_val.shape[1] == X_test.shape[1]

print("Data validation passed.")

Training: (15991, 50000) (15991,)
Validation: (1999, 50000) (1999,)
Testing: (1999, 50000) (1999,)
Data validation passed.


### Check sentiment distribution

In [7]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.0+cpu
CUDA available: False


In [8]:
print("Training sentiment distribution:")
print(y_train.value_counts())

print("\nValidation sentiment distribution:")
print(y_val.value_counts())

Training sentiment distribution:
sentiment
positive    11475
negative     2686
neutral      1830
Name: count, dtype: int64

Validation sentiment distribution:
sentiment
positive    1435
negative     336
neutral      228
Name: count, dtype: int64


### Define machine learning models

In [9]:
custom_weights = {
    "positive": 1.0,
    "negative": 1.6,
    "neutral": 2.0
}

classifiers = {
    "Linear SVM": LinearSVC(
        C=0.5,
        class_weight=custom_weights,
        random_state=42
    ),

    "Logistic Regression": LogisticRegression(
        C=1.5,
        max_iter=2000,
        class_weight=custom_weights,
        random_state=42
    ),

    "Balanced Logistic Regression": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "Naive Bayes": MultinomialNB(),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

### Train and compare models

In [10]:
results = []
trained_models = {}

for name, model in classifiers.items():

    print(f"Training {name}...")

    model.fit(X_train, y_train)

    predictions = model.predict(X_val)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_val,
        predictions,
        average="macro",
        zero_division=0
    )

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, predictions),
        "Macro Precision": precision,
        "Macro Recall": recall,
        "Macro F1": f1
    })

    trained_models[name] = model

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Macro F1",
    ascending=False
).reset_index(drop=True)

display(results_df)

Training Linear SVM...
Training Logistic Regression...
Training Balanced Logistic Regression...
Training Naive Bayes...
Training Random Forest...


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Balanced Logistic Regression,0.780390,0.639288,0.675823,0.654395
1,Logistic Regression,0.804902,0.672370,0.591460,0.617684
2,Linear SVM,0.811906,0.678372,0.594752,0.616885
3,Random Forest,0.782891,0.669182,0.517216,0.527244
4,Naive Bayes,0.723862,0.550330,0.345998,0.304294


### Select and save the best model

In [11]:
best_name = results_df.loc[0, "Model"]
best_model = trained_models[best_name]

joblib.dump(
    best_model,
    MODEL_DIR / "best_tfidf_model.joblib"
)

print("Best model:", best_name)
print("Best Macro F1:", results_df.loc[0, "Macro F1"])
print("Best model saved successfully.")

Best model: Balanced Logistic Regression
Best Macro F1: 0.6543952077014998
Best model saved successfully.


### Save Logistic Regression separately

In [12]:
logistic_model = trained_models.get("Logistic Regression")

if logistic_model is not None:
    joblib.dump(
        logistic_model,
        MODEL_DIR / "logistic_regression_tfidf.joblib"
    )

    print("Logistic Regression model saved.")

Logistic Regression model saved.


### Save validation results

In [13]:
results_path = OUTPUT_DIR / "traditional_validation_results.csv"

results_df.to_csv(
    results_path,
    index=False
)

print("Results saved to:", results_path)

Results saved to: d:\My_Project\Analyse\mobile-review-sentiment-analysis\outputs\evaluation\traditional_validation_results.csv


### Classification report

In [14]:
val_predictions = best_model.predict(X_val)

print("Best model:", best_name)

print("\nClassification Report:")

print(
    classification_report(
        y_val,
        val_predictions,
        zero_division=0
    )
)

Best model: Balanced Logistic Regression

Classification Report:
              precision    recall  f1-score   support

    negative       0.65      0.72      0.68       336
     neutral       0.36      0.46      0.41       228
    positive       0.91      0.85      0.88      1435

    accuracy                           0.78      1999
   macro avg       0.64      0.68      0.65      1999
weighted avg       0.80      0.78      0.79      1999



### Confusion matrix

In [15]:
labels = sorted(y_val.unique())

cm = confusion_matrix(
    y_val,
    val_predictions,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in labels],
    columns=[f"Predicted {label}" for label in labels]
)

display(cm_df)

,Predicted negative,Predicted neutral,Predicted positive
Actual negative,241,48,47
Actual neutral,49,106,73
Actual positive,83,139,1213
